In [1]:
import pandas as pd
import numpy as np

data_path = r"data"

state_totals = pd.read_csv(
    rf"{data_path}\qcew_state_totals_2015_2025.csv"
)

state_industry = pd.read_csv(
    rf"{data_path}\qcew_state_industry_2015_2025.csv"
)

oews = pd.read_csv(
    rf"{data_path}\oews_state_occupation_2021_2025.csv"
)

print("STATE TOTALS")
print(state_totals.shape)

print("\nSTATE INDUSTRY")
print(state_industry.shape)

print("\nOEWS")
print(oews.shape)

print("\nSTATE TOTALS COLUMNS")
print(state_totals.columns.tolist())

print("\nSTATE TOTALS YEARS")
print(state_totals["year"].min(), "-", state_totals["year"].max())

print("\nSTATES:", state_totals["state"].nunique())

STATE TOTALS
(561, 14)

STATE INDUSTRY
(11629, 11)

OEWS
(182774, 24)

STATE TOTALS COLUMNS
['year', 'area_fips', 'state', 'employment', 'establishments', 'total_wages', 'avg_annual_pay', 'avg_weekly_wage', 'oty_annual_avg_emplvl_chg', 'oty_annual_avg_emplvl_pct_chg', 'oty_total_annual_wages_chg', 'oty_total_annual_wages_pct_chg', 'oty_avg_annual_pay_chg', 'oty_avg_annual_pay_pct_chg']

STATE TOTALS YEARS
2015 - 2025

STATES: 51


In [2]:
# ==========================================
# STATE EDA 1
# Long-term change: 2015 → 2025
# ==========================================

state_2015 = (
    state_totals[state_totals["year"] == 2015]
    [["state", "employment", "avg_annual_pay"]]
    .rename(columns={
        "employment": "employment_2015",
        "avg_annual_pay": "pay_2015"
    })
)

state_2025 = (
    state_totals[state_totals["year"] == 2025]
    [["state", "employment", "avg_annual_pay"]]
    .rename(columns={
        "employment": "employment_2025",
        "avg_annual_pay": "pay_2025"
    })
)

state_change = state_2015.merge(
    state_2025,
    on="state",
    how="inner"
)

# Long-term percentage changes
state_change["employment_growth_pct"] = (
    (state_change["employment_2025"] - state_change["employment_2015"])
    / state_change["employment_2015"]
    * 100
)

state_change["pay_growth_pct"] = (
    (state_change["pay_2025"] - state_change["pay_2015"])
    / state_change["pay_2015"]
    * 100
)

print("States:", len(state_change))

print("\n=== EMPLOYMENT GROWTH 2015–2025 ===")
print(
    state_change["employment_growth_pct"]
    .describe()
    .round(2)
)

print("\nTOP 10:")
print(
    state_change[
        ["state", "employment_growth_pct"]
    ]
    .sort_values("employment_growth_pct", ascending=False)
    .head(10)
    .round(2)
    .to_string(index=False)
)

print("\nBOTTOM 10:")
print(
    state_change[
        ["state", "employment_growth_pct"]
    ]
    .sort_values("employment_growth_pct")
    .head(10)
    .round(2)
    .to_string(index=False)
)

print("\n=== PAY GROWTH 2015–2025 ===")
print(
    state_change["pay_growth_pct"]
    .describe()
    .round(2)
)

print("\nTOP 10:")
print(
    state_change[
        ["state", "pay_growth_pct"]
    ]
    .sort_values("pay_growth_pct", ascending=False)
    .head(10)
    .round(2)
    .to_string(index=False)
)

print("\nBOTTOM 10:")
print(
    state_change[
        ["state", "pay_growth_pct"]
    ]
    .sort_values("pay_growth_pct")
    .head(10)
    .round(2)
    .to_string(index=False)
)

States: 51

=== EMPLOYMENT GROWTH 2015–2025 ===
count    51.00
mean      9.74
std       7.97
min      -1.72
25%       5.27
50%       8.44
75%      13.15
max      31.51
Name: employment_growth_pct, dtype: float64

TOP 10:
         state  employment_growth_pct
         Idaho                  31.51
          Utah                  29.87
        Nevada                  26.08
       Arizona                  23.95
       Florida                  23.32
         Texas                  21.05
South Carolina                  19.03
North Carolina                  18.74
       Georgia                  17.81
      Colorado                  15.92

BOTTOM 10:
               state  employment_growth_pct
        North Dakota                  -1.72
District of Columbia                  -0.26
             Wyoming                  -0.15
           Louisiana                  -0.14
       West Virginia                  -0.11
              Alaska                   0.26
             Vermont                   0.

In [3]:
# ==========================================
# STATE EDA 2
# Employment growth vs Pay growth
# ==========================================

correlation = state_change[
    ["employment_growth_pct", "pay_growth_pct"]
].corr().iloc[0, 1]

print("Correlation between employment growth and pay growth:")
print(round(correlation, 3))


# Median benchmarks
employment_median = state_change["employment_growth_pct"].median()
pay_median = state_change["pay_growth_pct"].median()

print("\nMedian employment growth:", round(employment_median, 2))
print("Median pay growth:", round(pay_median, 2))


# Classify states into four groups
def classify_state(row):

    high_emp = row["employment_growth_pct"] >= employment_median
    high_pay = row["pay_growth_pct"] >= pay_median

    if high_emp and high_pay:
        return "High Employment / High Pay"

    elif high_emp and not high_pay:
        return "High Employment / Low Pay"

    elif not high_emp and high_pay:
        return "Low Employment / High Pay"

    else:
        return "Low Employment / Low Pay"


state_change["growth_pattern"] = state_change.apply(
    classify_state,
    axis=1
)


print("\n=== PATTERN COUNTS ===")
print(
    state_change["growth_pattern"]
    .value_counts()
)


print("\n=== STATES BY PATTERN ===")

for pattern in [
    "High Employment / High Pay",
    "High Employment / Low Pay",
    "Low Employment / High Pay",
    "Low Employment / Low Pay"
]:

    print(f"\n{pattern}")

    print(
        state_change.loc[
            state_change["growth_pattern"] == pattern,
            [
                "state",
                "employment_growth_pct",
                "pay_growth_pct"
            ]
        ]
        .sort_values(
            "employment_growth_pct",
            ascending=False
        )
        .round(2)
        .to_string(index=False)
    )

Correlation between employment growth and pay growth:
0.602

Median employment growth: 8.44
Median pay growth: 47.16

=== PATTERN COUNTS ===
growth_pattern
High Employment / High Pay    21
Low Employment / Low Pay      20
High Employment / Low Pay      5
Low Employment / High Pay      5
Name: count, dtype: int64

=== STATES BY PATTERN ===

High Employment / High Pay
         state  employment_growth_pct  pay_growth_pct
         Idaho                  31.51           60.57
          Utah                  29.87           56.41
        Nevada                  26.08           52.55
       Arizona                  23.95           52.07
       Florida                  23.32           57.31
South Carolina                  19.03           51.73
North Carolina                  18.74           53.74
       Georgia                  17.81           47.39
      Colorado                  15.92           55.32
     Tennessee                  15.79           50.63
    Washington                  15.34

In [4]:
# ==========================================
# STATE EDA 3
# Examine annual paths for selected states
# ==========================================

selected_states = [
    "Idaho",
    "Utah",
    "Florida",
    "Washington",
    "Texas",
    "Hawaii",
    "North Dakota"
]

state_trends = (
    state_totals[
        state_totals["state"].isin(selected_states)
    ]
    [
        [
            "year",
            "state",
            "employment",
            "avg_annual_pay",
            "oty_annual_avg_emplvl_pct_chg",
            "oty_avg_annual_pay_pct_chg"
        ]
    ]
    .sort_values(["state", "year"])
)

print("=== ANNUAL EMPLOYMENT GROWTH (%) ===")

print(
    state_trends.pivot(
        index="year",
        columns="state",
        values="oty_annual_avg_emplvl_pct_chg"
    )
    .round(2)
    .to_string()
)


print("\n=== ANNUAL PAY GROWTH (%) ===")

print(
    state_trends.pivot(
        index="year",
        columns="state",
        values="oty_avg_annual_pay_pct_chg"
    )
    .round(2)
    .to_string()
)

=== ANNUAL EMPLOYMENT GROWTH (%) ===
state  Florida  Hawaii  Idaho  North Dakota  Texas  Utah  Washington
year                                                                
2015       3.7     1.9    2.9          -1.7    2.4   3.8         2.6
2016       3.4     1.5    3.5          -4.6    1.3   3.6         3.0
2017       2.2     1.0    2.7          -0.7    1.8   3.0         2.3
2018       2.4     0.6    3.4           0.9    2.4   3.3         2.5
2019       2.1     0.1    3.0           1.3    2.3   2.9         2.0
2020      -4.9   -15.0   -0.6          -6.4   -4.1  -1.0        -5.3
2021       4.9     5.0    5.7           1.1    3.8   5.2         2.9
2022       5.6     5.4    3.9           2.7    5.8   4.3         4.8
2023       3.4     2.2    2.6           2.5    3.4   2.4         1.9
2024       1.6     0.9    1.9           1.7    1.7   1.4         0.5
2025       0.8     0.7    1.8           0.2    1.2   1.6         0.2

=== ANNUAL PAY GROWTH (%) ===
state  Florida  Hawaii  Idaho  Nort

In [5]:
# ==========================================
# INDUSTRY EDA 0
# Suppression structure
# ==========================================

# 1. Overall suppression
print("=== OVERALL SUPPRESSION ===")

print(
    state_industry["has_suppressed_data"]
    .value_counts(dropna=False)
)

print(
    "\nSuppression rate:",
    round(
        state_industry["has_suppressed_data"].mean() * 100,
        2
    ),
    "%"
)


# ------------------------------------------
# 2. Suppression by year
# ------------------------------------------

print("\n=== SUPPRESSION RATE BY YEAR ===")

suppression_year = (
    state_industry
    .groupby("year")["has_suppressed_data"]
    .mean()
    .mul(100)
    .round(2)
)

print(suppression_year)


# ------------------------------------------
# 3. Suppression by state
# ------------------------------------------

print("\n=== STATES WITH HIGHEST SUPPRESSION ===")

suppression_state = (
    state_industry
    .groupby("state")["has_suppressed_data"]
    .agg(
        rows="size",
        suppressed="sum",
        rate="mean"
    )
)

suppression_state["rate"] *= 100

print(
    suppression_state
    .sort_values("rate", ascending=False)
    .head(15)
    .round(2)
    .to_string()
)


print("\n=== STATES WITH LOWEST SUPPRESSION ===")

print(
    suppression_state
    .sort_values("rate")
    .head(15)
    .round(2)
    .to_string()
)


# ------------------------------------------
# 4. Selected states from our first EDA
# ------------------------------------------

selected_states = [
    "Idaho",
    "Utah",
    "Florida",
    "Washington",
    "Texas",
    "Hawaii",
    "North Dakota"
]

print("\n=== SELECTED STATES: SUPPRESSION BY YEAR ===")

selected_suppression = (
    state_industry[
        state_industry["state"].isin(selected_states)
    ]
    .groupby(["year", "state"])["has_suppressed_data"]
    .mean()
    .mul(100)
    .unstack()
    .round(1)
)

print(selected_suppression.to_string())

=== OVERALL SUPPRESSION ===
has_suppressed_data
False    7849
True     3780
Name: count, dtype: int64

Suppression rate: 32.5 %

=== SUPPRESSION RATE BY YEAR ===
year
2015    37.86
2016    37.51
2017    38.63
2018    39.02
2019    39.09
2020    38.90
2021    38.81
2022    17.61
2023    17.37
2024    19.62
2025    33.46
Name: has_suppressed_data, dtype: float64

=== STATES WITH HIGHEST SUPPRESSION ===
                rows  suppressed   rate
state                                  
South Dakota     224         125  55.80
Louisiana        231         117  50.65
New York         231         112  48.48
Wyoming          228         109  47.81
Delaware         224         105  46.88
Massachusetts    225         105  46.67
South Carolina   228         105  46.05
Iowa             222         101  45.50
Arizona          231         103  44.59
Maryland         229         102  44.54
Alabama          224          97  43.30
Vermont          224          92  41.07
Rhode Island     231          94  40

In [6]:
# ==========================================
# INDUSTRY EDA 1
# Employment coverage vs official state totals
# ==========================================

industry_coverage = (
    state_industry
    .groupby(["year", "state"], as_index=False)
    ["employment"]
    .sum()
    .rename(columns={
        "employment": "industry_published_employment"
    })
)

official_employment = (
    state_totals[
        ["year", "state", "employment"]
    ]
    .rename(columns={
        "employment": "official_state_employment"
    })
)

coverage = industry_coverage.merge(
    official_employment,
    on=["year", "state"],
    how="left"
)

coverage["employment_coverage_pct"] = (
    coverage["industry_published_employment"]
    / coverage["official_state_employment"]
    * 100
)


print("=== COVERAGE DISTRIBUTION ===")

print(
    coverage["employment_coverage_pct"]
    .describe()
    .round(2)
)


print("\n=== COVERAGE BY YEAR ===")

print(
    coverage
    .groupby("year")["employment_coverage_pct"]
    .agg(["mean", "median", "min", "max"])
    .round(2)
)


print("\n=== LOWEST COVERAGE OBSERVATIONS ===")

print(
    coverage[
        [
            "year",
            "state",
            "official_state_employment",
            "industry_published_employment",
            "employment_coverage_pct"
        ]
    ]
    .sort_values("employment_coverage_pct")
    .head(20)
    .round(2)
    .to_string(index=False)
)


selected_states = [
    "Idaho",
    "Utah",
    "Florida",
    "Washington",
    "Texas",
    "Hawaii",
    "North Dakota"
]

print("\n=== SELECTED STATES COVERAGE ===")

print(
    coverage[
        coverage["state"].isin(selected_states)
    ]
    .pivot(
        index="year",
        columns="state",
        values="employment_coverage_pct"
    )
    .round(1)
    .to_string()
)

=== COVERAGE DISTRIBUTION ===
count    561.00
mean      98.92
std        3.84
min       52.84
25%       99.60
50%       99.82
75%       99.98
max      100.00
Name: employment_coverage_pct, dtype: float64

=== COVERAGE BY YEAR ===
       mean  median    min     max
year                              
2015  99.54   99.77  97.53  100.00
2016  99.62   99.78  97.54  100.00
2017  99.52   99.81  95.84  100.00
2018  99.64   99.77  97.58  100.00
2019  99.60   99.76  97.65  100.00
2020  99.53   99.77  95.07  100.00
2021  99.26   99.77  90.97   99.99
2022  99.55  100.00  90.78  100.00
2023  99.59  100.00  94.47  100.00
2024  99.33  100.00  94.32  100.00
2025  92.89   99.62  52.84  100.00

=== LOWEST COVERAGE OBSERVATIONS ===
 year                state  official_state_employment  industry_published_employment  employment_coverage_pct
 2025              Wyoming                     282252                         149133                    52.84
 2025              Vermont                     308225    

In [7]:
# ==========================================
# INDUSTRY EDA 2
# Check industry completeness for selected
# high-growth states: 2015 vs 2025
# ==========================================

driver_states = [
    "Florida",
    "Idaho",
    "Texas"
]

industry_endpoints = state_industry[
    (state_industry["state"].isin(driver_states)) &
    (state_industry["year"].isin([2015, 2025]))
].copy()


print("=== SUPPRESSED INDUSTRIES: 2015 / 2025 ===")

for state in driver_states:

    print(f"\n--- {state} ---")

    temp = industry_endpoints[
        (industry_endpoints["state"] == state) &
        (industry_endpoints["has_suppressed_data"])
    ][
        [
            "year",
            "industry_code",
            "industry",
            "employment"
        ]
    ].sort_values(["year", "industry_code"])

    if len(temp) == 0:
        print("No suppressed industries.")
    else:
        print(temp.to_string(index=False))

=== SUPPRESSED INDUSTRIES: 2015 / 2025 ===

--- Florida ---
 year industry_code                                                                 industry  employment
 2015            11                               Agriculture, Forestry, Fishing and Hunting       76205
 2015            23                                                             Construction      431413
 2015         48-49                                           Transportation and Warehousing      278259
 2015            51                                                              Information      136175
 2015            54                         Professional, Scientific, and Technical Services      509378
 2015            56 Administrative and Support and Waste Management and Remediation Services      618674
 2015            81                            Other Services (except Public Administration)      262305
 2015            99                                                             Unclassified        

In [8]:
# ==========================================
# INDUSTRY EDA 3
# Quantify total missing employment
# ==========================================

selected_coverage = coverage[
    coverage["state"].isin([
        "Florida",
        "Idaho",
        "Texas",
        "North Dakota",
        "Utah",
        "Washington"
    ])
].copy()

selected_coverage["missing_employment"] = (
    selected_coverage["official_state_employment"]
    - selected_coverage["industry_published_employment"]
)

selected_coverage["missing_pct"] = (
    selected_coverage["missing_employment"]
    / selected_coverage["official_state_employment"]
    * 100
)

print(
    selected_coverage[
        selected_coverage["year"].isin([2015, 2025])
    ][
        [
            "year",
            "state",
            "official_state_employment",
            "industry_published_employment",
            "missing_employment",
            "missing_pct"
        ]
    ]
    .sort_values(["state", "year"])
    .round(2)
    .to_string(index=False)
)

 year        state  official_state_employment  industry_published_employment  missing_employment  missing_pct
 2015      Florida                    8039635                        8031228                8407         0.10
 2025      Florida                    9914411                        9914413                  -2        -0.00
 2015        Idaho                     664792                         663580                1212         0.18
 2025        Idaho                     874283                         874286                  -3        -0.00
 2015 North Dakota                     437072                         430577                6495         1.49
 2025 North Dakota                     429567                         351087               78480        18.27
 2015        Texas                   11655919                       11655848                  71         0.00
 2025        Texas                   14109957                       14106870                3087         0.02
 2015     

In [9]:
# ==========================================
# INDUSTRY EDA 4
# Industry contribution to employment growth
# Florida, Idaho, Texas
# 2015 -> 2025
# ==========================================

driver_states = [
    "Florida",
    "Idaho",
    "Texas"
]

industry_2015 = (
    state_industry[
        (state_industry["year"] == 2015) &
        (state_industry["state"].isin(driver_states))
    ]
    [
        [
            "state",
            "industry_code",
            "industry",
            "employment",
            "has_suppressed_data"
        ]
    ]
    .rename(columns={
        "employment": "employment_2015",
        "has_suppressed_data": "suppressed_2015"
    })
)

industry_2025 = (
    state_industry[
        (state_industry["year"] == 2025) &
        (state_industry["state"].isin(driver_states))
    ]
    [
        [
            "state",
            "industry_code",
            "industry",
            "employment",
            "has_suppressed_data"
        ]
    ]
    .rename(columns={
        "employment": "employment_2025",
        "has_suppressed_data": "suppressed_2025"
    })
)

industry_growth = industry_2015.merge(
    industry_2025,
    on=["state", "industry_code", "industry"],
    how="inner"
)

industry_growth["jobs_added"] = (
    industry_growth["employment_2025"]
    - industry_growth["employment_2015"]
)

industry_growth["industry_growth_pct"] = (
    industry_growth["jobs_added"]
    / industry_growth["employment_2015"]
    * 100
)


# Official state employment growth
state_growth_lookup = (
    state_change[
        state_change["state"].isin(driver_states)
    ]
    .set_index("state")
)

industry_growth["state_jobs_added"] = (
    industry_growth["state"].map(
        state_growth_lookup["employment_2025"]
        - state_growth_lookup["employment_2015"]
    )
)

industry_growth["contribution_pct"] = (
    industry_growth["jobs_added"]
    / industry_growth["state_jobs_added"]
    * 100
)


# ------------------------------------------
# Print results
# ------------------------------------------

for state in driver_states:

    print(f"\n{'=' * 70}")
    print(state.upper())
    print("=" * 70)

    temp = (
        industry_growth[
            industry_growth["state"] == state
        ]
        [
            [
                "industry",
                "employment_2015",
                "employment_2025",
                "jobs_added",
                "industry_growth_pct",
                "contribution_pct",
                "suppressed_2015",
                "suppressed_2025"
            ]
        ]
        .sort_values("jobs_added", ascending=False)
    )

    print(
        temp
        .round(2)
        .to_string(index=False)
    )


FLORIDA
                                                                industry  employment_2015  employment_2025  jobs_added  industry_growth_pct  contribution_pct  suppressed_2015  suppressed_2025
                                       Health Care and Social Assistance          1118276          1451924      333648                29.84             17.80            False            False
                        Professional, Scientific, and Technical Services           509378           765837      256459                50.35             13.68             True            False
                                                            Construction           431413           659557      228144                52.88             12.17             True            False
                                          Transportation and Warehousing           278259           456390      178131                64.02              9.50             True            False
                               

In [10]:
# ==========================================
# INDUSTRY EDA 5
# Systematic industry growth drivers
# across states
# ==========================================

# ------------------------------------------
# 1. Identify states with >=95% employment
# coverage in BOTH 2015 and 2025
# ------------------------------------------

coverage_endpoints = (
    coverage[
        coverage["year"].isin([2015, 2025])
    ]
    .pivot(
        index="state",
        columns="year",
        values="employment_coverage_pct"
    )
)

valid_states = coverage_endpoints[
    (coverage_endpoints[2015] >= 95) &
    (coverage_endpoints[2025] >= 95)
].index.tolist()

excluded_states = coverage_endpoints.index.difference(valid_states).tolist()

print("Valid states:", len(valid_states))
print("Excluded states:", len(excluded_states))
print("\nExcluded:")
print(excluded_states)


# ------------------------------------------
# 2. Build 2015 -> 2025 industry growth
# ------------------------------------------

ind_2015 = (
    state_industry[
        (state_industry["year"] == 2015) &
        (state_industry["state"].isin(valid_states))
    ]
    [
        ["state", "industry_code", "industry", "employment"]
    ]
    .rename(columns={"employment": "employment_2015"})
)

ind_2025 = (
    state_industry[
        (state_industry["year"] == 2025) &
        (state_industry["state"].isin(valid_states))
    ]
    [
        ["state", "industry_code", "industry", "employment"]
    ]
    .rename(columns={"employment": "employment_2025"})
)

all_industry_growth = ind_2015.merge(
    ind_2025,
    on=["state", "industry_code", "industry"],
    how="inner"
)

all_industry_growth["jobs_added"] = (
    all_industry_growth["employment_2025"]
    - all_industry_growth["employment_2015"]
)


# ------------------------------------------
# 3. Rank industries within each state
# ------------------------------------------

all_industry_growth["driver_rank"] = (
    all_industry_growth
    .groupby("state")["jobs_added"]
    .rank(
        method="first",
        ascending=False
    )
)


# ------------------------------------------
# 4. Count how often each industry appears
# among Top 3 growth drivers
# ------------------------------------------

top3 = all_industry_growth[
    all_industry_growth["driver_rank"] <= 3
].copy()

top3_frequency = (
    top3
    .groupby("industry")
    .agg(
        states_top3=("state", "nunique"),
        total_jobs_added=("jobs_added", "sum"),
        median_jobs_added=("jobs_added", "median")
    )
    .sort_values(
        ["states_top3", "total_jobs_added"],
        ascending=False
    )
)

print("\n=== INDUSTRIES MOST OFTEN IN TOP 3 STATE GROWTH DRIVERS ===")

print(
    top3_frequency
    .round(0)
    .to_string()
)


# ------------------------------------------
# 5. Most common #1 driver
# ------------------------------------------

number_one = all_industry_growth[
    all_industry_growth["driver_rank"] == 1
]

number_one_frequency = (
    number_one
    .groupby("industry")["state"]
    .nunique()
    .sort_values(ascending=False)
)

print("\n=== MOST COMMON #1 EMPLOYMENT GROWTH DRIVER ===")

print(number_one_frequency)


# ------------------------------------------
# 6. Show each state's #1 driver
# ------------------------------------------

print("\n=== #1 DRIVER BY STATE ===")

print(
    number_one[
        ["state", "industry", "jobs_added"]
    ]
    .sort_values("state")
    .to_string(index=False)
)

Valid states: 29
Excluded states: 22

Excluded:
['Alaska', 'Arkansas', 'Connecticut', 'Delaware', 'District of Columbia', 'Hawaii', 'Indiana', 'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 'Missouri', 'Nebraska', 'North Carolina', 'North Dakota', 'Rhode Island', 'South Carolina', 'South Dakota', 'Utah', 'Vermont', 'Wisconsin', 'Wyoming']

=== INDUSTRIES MOST OFTEN IN TOP 3 STATE GROWTH DRIVERS ===
                                                  states_top3  total_jobs_added  median_jobs_added
industry                                                                                          
Health Care and Social Assistance                          29           4001940            82698.0
Professional, Scientific, and Technical Services           18           1246247            41513.0
Transportation and Warehousing                             17           1229766            62280.0
Construction                                               15            715836            30263.0


In [11]:
# ==========================================
# INDUSTRY EDA 6
# Robustness check: 2015 -> 2024
# ==========================================

start_year = 2015
end_year = 2024
coverage_threshold = 95


# ------------------------------------------
# 1. Valid states
# ------------------------------------------

coverage_check = (
    coverage[
        coverage["year"].isin([start_year, end_year])
    ]
    .pivot(
        index="state",
        columns="year",
        values="employment_coverage_pct"
    )
)

valid_states_2024 = coverage_check[
    (coverage_check[start_year] >= coverage_threshold) &
    (coverage_check[end_year] >= coverage_threshold)
].index.tolist()

excluded_states_2024 = (
    coverage_check.index
    .difference(valid_states_2024)
    .tolist()
)

print("Valid states:", len(valid_states_2024))
print("Excluded states:", len(excluded_states_2024))
print("Excluded:", excluded_states_2024)


# ------------------------------------------
# 2. Build industry growth
# ------------------------------------------

start_data = (
    state_industry[
        (state_industry["year"] == start_year) &
        (state_industry["state"].isin(valid_states_2024))
    ]
    [["state", "industry_code", "industry", "employment"]]
    .rename(columns={"employment": "employment_start"})
)

end_data = (
    state_industry[
        (state_industry["year"] == end_year) &
        (state_industry["state"].isin(valid_states_2024))
    ]
    [["state", "industry_code", "industry", "employment"]]
    .rename(columns={"employment": "employment_end"})
)

growth_2024 = start_data.merge(
    end_data,
    on=["state", "industry_code", "industry"],
    how="inner"
)

growth_2024["jobs_added"] = (
    growth_2024["employment_end"]
    - growth_2024["employment_start"]
)


# ------------------------------------------
# 3. Rank drivers within each state
# ------------------------------------------

growth_2024["driver_rank"] = (
    growth_2024
    .groupby("state")["jobs_added"]
    .rank(method="first", ascending=False)
)


# ------------------------------------------
# 4. Top 3 frequency
# ------------------------------------------

top3_2024 = growth_2024[
    growth_2024["driver_rank"] <= 3
]

top3_frequency_2024 = (
    top3_2024
    .groupby("industry")
    .agg(
        states_top3=("state", "nunique"),
        total_jobs_added=("jobs_added", "sum")
    )
    .sort_values(
        ["states_top3", "total_jobs_added"],
        ascending=False
    )
)

print("\n=== TOP 3 DRIVER FREQUENCY: 2015-2024 ===")

print(
    top3_frequency_2024
    .round(0)
    .to_string()
)


# ------------------------------------------
# 5. #1 driver frequency
# ------------------------------------------

number_one_2024 = growth_2024[
    growth_2024["driver_rank"] == 1
]

number_one_frequency_2024 = (
    number_one_2024
    .groupby("industry")["state"]
    .nunique()
    .sort_values(ascending=False)
)

print("\n=== #1 DRIVER FREQUENCY: 2015-2024 ===")

print(number_one_frequency_2024)


# ------------------------------------------
# 6. States where Healthcare is NOT #1
# ------------------------------------------

print("\n=== STATES WHERE HEALTHCARE IS NOT #1 ===")

print(
    number_one_2024[
        number_one_2024["industry"]
        != "Health Care and Social Assistance"
    ][
        ["state", "industry", "jobs_added"]
    ]
    .sort_values("state")
    .to_string(index=False)
)

Valid states: 50
Excluded states: 1
Excluded: ['Delaware']

=== TOP 3 DRIVER FREQUENCY: 2015-2024 ===
                                                                          states_top3  total_jobs_added
industry                                                                                               
Health Care and Social Assistance                                                  47           4001760
Professional, Scientific, and Technical Services                                   33           1847913
Construction                                                                       24            970630
Transportation and Warehousing                                                     20           1236954
Accommodation and Food Services                                                     8            345610
Manufacturing                                                                       4             67172
Arts, Entertainment, and Recreation                               

In [12]:
# ==========================================
# OCCUPATION EDA 1
# Major occupational groups - 2025
# ==========================================

major_2025 = oews[
    (oews["year"] == 2025) &
    (oews["occupation_level"] == "major")
].copy()


print("Rows:", len(major_2025))
print("States:", major_2025["state"].nunique())
print("Occupational groups:", major_2025["occupation_code"].nunique())


# ------------------------------------------
# National view:
# total employment across states by group
# ------------------------------------------

occupation_summary = (
    major_2025
    .groupby(["occupation_code", "occupation"])
    .agg(
        total_employment=("employment", "sum"),
        median_state_lq=("location_quotient", "median"),
        min_state_lq=("location_quotient", "min"),
        max_state_lq=("location_quotient", "max")
    )
    .sort_values("total_employment", ascending=False)
)

print("\n=== MAJOR OCCUPATIONAL GROUPS ===")

print(
    occupation_summary
    .round(2)
    .to_string()
)

Rows: 1122
States: 51
Occupational groups: 22

=== MAJOR OCCUPATIONAL GROUPS ===
                                                                            total_employment  median_state_lq  min_state_lq  max_state_lq
occupation_code occupation                                                                                                               
43-0000         Office and Administrative Support Occupations                     17753410.0             1.01          0.67          1.14
35-0000         Food Preparation and Serving Related Occupations                  13680740.0             0.99          0.84          1.50
53-0000         Transportation and Material Moving Occupations                    13677150.0             0.99          0.30          1.28
41-0000         Sales and Related Occupations                                     13424290.0             0.99          0.34          1.23
11-0000         Management Occupations                                            11132660.

In [13]:
# ==========================================
# OCCUPATION EDA 2
# Highest occupational concentrations
# by state - 2025
# ==========================================

# Remove missing LQ values
major_lq = major_2025.dropna(
    subset=["location_quotient"]
).copy()


# ------------------------------------------
# Top 3 states for each occupational group
# ------------------------------------------

top_lq_states = (
    major_lq
    .sort_values(
        ["occupation", "location_quotient"],
        ascending=[True, False]
    )
    .groupby(
        ["occupation_code", "occupation"],
        group_keys=False
    )
    .head(3)
)


for occupation in occupation_summary.index:

    code, title = occupation

    temp = top_lq_states[
        top_lq_states["occupation_code"] == code
    ][
        [
            "state",
            "location_quotient",
            "employment",
            "mean_annual_wage"
        ]
    ]

    print(f"\n{title}")

    print(
        temp
        .round(2)
        .to_string(index=False)
    )


Office and Administrative Support Occupations
     state  location_quotient  employment  mean_annual_wage
 Tennessee               1.14    426040.0           48790.0
New Mexico               1.13    112620.0           47230.0
      Utah               1.12    221760.0           49040.0

Food Preparation and Serving Related Occupations
  state  location_quotient  employment  mean_annual_wage
 Hawaii               1.50     82670.0           49360.0
 Nevada               1.50    204340.0           37500.0
Montana               1.18     53110.0           33550.0

Transportation and Material Moving Occupations
    state  location_quotient  employment  mean_annual_wage
  Indiana               1.28    359500.0           47500.0
 Kentucky               1.28    225530.0           54280.0
Tennessee               1.28    369270.0           46300.0

Sales and Related Occupations
        state  location_quotient  employment  mean_annual_wage
      Florida               1.23   1055160.0           52

In [14]:
# ==========================================
# OCCUPATION EDA 3
# Occupational concentration vs pay
# across states - 2025
# ==========================================

occupation_lq_pay = (
    major_2025
    .dropna(
        subset=[
            "location_quotient",
            "mean_annual_wage"
        ]
    )
    .copy()
)


# ------------------------------------------
# Correlation within each occupation
# ------------------------------------------

lq_pay_relationship = (
    occupation_lq_pay
    .groupby(
        ["occupation_code", "occupation"]
    )
    .apply(
        lambda x: pd.Series({
            "states": len(x),
            "lq_pay_correlation":
                x["location_quotient"]
                .corr(x["mean_annual_wage"]),

            "median_lq":
                x["location_quotient"].median(),

            "median_wage":
                x["mean_annual_wage"].median()
        }),
        include_groups=False
    )
    .reset_index()
    .sort_values(
        "lq_pay_correlation",
        ascending=False
    )
)


print(
    lq_pay_relationship
    .round(3)
    .to_string(index=False)
)

occupation_code                                                 occupation  states  lq_pay_correlation  median_lq  median_wage
        15-0000                      Computer and Mathematical Occupations    51.0               0.845       0.84     104140.0
        13-0000              Business and Financial Operations Occupations    51.0               0.763       0.92      86090.0
        27-0000 Arts, Design, Entertainment, Sports, and Media Occupations    51.0               0.747       0.87      66610.0
        23-0000                                          Legal Occupations    51.0               0.630       0.86     116610.0
        17-0000                   Architecture and Engineering Occupations    51.0               0.547       0.98      99940.0
        19-0000             Life, Physical, and Social Science Occupations    51.0               0.531       0.97      85220.0
        21-0000                   Community and Social Service Occupations    51.0               0.368       0.

In [15]:
# ==========================================
# OCCUPATION EDA 4
# Detailed occupations quality - 2025
# ==========================================

detailed_2025 = oews[
    (oews["year"] == 2025) &
    (oews["occupation_level"] == "detailed")
].copy()


print("=== OVERALL STRUCTURE ===")

print("Rows:", len(detailed_2025))
print("States:", detailed_2025["state"].nunique())
print(
    "Unique detailed occupations:",
    detailed_2025["occupation_code"].nunique()
)


# ------------------------------------------
# Missing values
# ------------------------------------------

print("\n=== MISSING VALUES ===")

for column in [
    "employment",
    "location_quotient",
    "mean_annual_wage",
    "median_annual_wage"
]:
    missing = detailed_2025[column].isna().sum()
    pct = missing / len(detailed_2025) * 100

    print(
        f"{column}: {missing} "
        f"({pct:.2f}%)"
    )


# ------------------------------------------
# Availability by state
# ------------------------------------------

state_detail_quality = (
    detailed_2025
    .groupby("state")
    .agg(
        occupation_rows=("occupation_code", "size"),
        employment_available=("employment", "count"),
        wage_available=("mean_annual_wage", "count"),
        lq_available=("location_quotient", "count")
    )
)

state_detail_quality["employment_coverage_pct"] = (
    state_detail_quality["employment_available"]
    / state_detail_quality["occupation_rows"]
    * 100
)

state_detail_quality["wage_coverage_pct"] = (
    state_detail_quality["wage_available"]
    / state_detail_quality["occupation_rows"]
    * 100
)


print("\n=== STATE QUALITY SUMMARY ===")

print(
    state_detail_quality[
        [
            "occupation_rows",
            "employment_coverage_pct",
            "wage_coverage_pct"
        ]
    ]
    .describe()
    .round(2)
)


print("\n=== LOWEST EMPLOYMENT AVAILABILITY ===")

print(
    state_detail_quality
    .sort_values("employment_coverage_pct")
    .head(10)
    .round(2)
    .to_string()
)


print("\n=== LOWEST WAGE AVAILABILITY ===")

print(
    state_detail_quality
    .sort_values("wage_coverage_pct")
    .head(10)
    .round(2)
    .to_string()
)

=== OVERALL STRUCTURE ===
Rows: 35223
States: 51
Unique detailed occupations: 829

=== MISSING VALUES ===
employment: 1281 (3.64%)
location_quotient: 1281 (3.64%)
mean_annual_wage: 699 (1.98%)
median_annual_wage: 699 (1.98%)

=== STATE QUALITY SUMMARY ===
       occupation_rows  employment_coverage_pct  wage_coverage_pct
count            51.00                    51.00              51.00
mean            690.65                    96.32              97.98
std              86.17                     1.53               0.72
min             475.00                    91.70              95.36
25%             641.00                    95.96              97.59
50%             718.00                    96.57              98.17
75%             753.00                    97.37              98.50
max             807.00                    98.59              99.00

=== LOWEST EMPLOYMENT AVAILABILITY ===
                      occupation_rows  employment_available  wage_available  lq_available  employment

In [16]:
# ==========================================
# OCCUPATION EDA 5
# Drill-down example:
# Washington - Computer & Mathematical
# ==========================================

wa_computer = detailed_2025[
    (detailed_2025["state"] == "Washington") &
    (detailed_2025["occupation_code"].str.startswith("15-"))
].copy()


print("Detailed occupations:", len(wa_computer))


print("\n=== LARGEST COMPUTER OCCUPATIONS ===")

print(
    wa_computer[
        [
            "occupation",
            "employment",
            "location_quotient",
            "mean_annual_wage"
        ]
    ]
    .sort_values("employment", ascending=False)
    .head(15)
    .round(2)
    .to_string(index=False)
)


print("\n=== MOST SPECIALIZED COMPUTER OCCUPATIONS ===")

print(
    wa_computer[
        [
            "occupation",
            "employment",
            "location_quotient",
            "mean_annual_wage"
        ]
    ]
    .dropna(subset=["location_quotient"])
    .sort_values("location_quotient", ascending=False)
    .head(15)
    .round(2)
    .to_string(index=False)
)

Detailed occupations: 21

=== LARGEST COMPUTER OCCUPATIONS ===
                                     occupation  employment  location_quotient  mean_annual_wage
                            Software Developers    107030.0               2.78          171040.0
                      Computer Systems Analysts     24120.0               2.03          131270.0
              Computer User Support Specialists     21860.0               1.33           80340.0
                Computer Occupations, All Other     14360.0               1.44          129500.0
            Web and Digital Interface Designers     11580.0               4.47          134700.0
                                Data Scientists      9600.0               1.60          158040.0
    Network and Computer Systems Administrators      8600.0               1.20          107640.0
Software Quality Assurance Analysts and Testers      7970.0               1.87          131090.0
                  Information Security Analysts      6030.0     

In [17]:
# ==========================================
# FINAL TABLEAU READINESS CHECK
# ==========================================

datasets = {
    "STATE TOTALS": state_totals,
    "STATE INDUSTRY": state_industry,
    "OEWS": oews
}

for name, df in datasets.items():

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print("Shape:", df.shape)

    # --------------------------------------
    # Duplicate rows
    # --------------------------------------
    print("\nExact duplicate rows:")
    print(df.duplicated().sum())

    # --------------------------------------
    # Column data types
    # --------------------------------------
    print("\nData types:")
    print(df.dtypes)

    # --------------------------------------
    # Missing values
    # --------------------------------------
    missing = (
        df.isna()
        .sum()
        .to_frame("missing")
    )

    missing["pct"] = (
        missing["missing"] / len(df) * 100
    )

    missing = missing[
        missing["missing"] > 0
    ]

    print("\nColumns with missing values:")

    if len(missing) == 0:
        print("None")
    else:
        print(missing.round(2))

    # --------------------------------------
    # Year range
    # --------------------------------------
    print("\nYear range:")
    print(df["year"].min(), "-", df["year"].max())


# ==========================================
# KEY DUPLICATE CHECKS
# ==========================================

print("\n" + "=" * 60)
print("KEY DUPLICATE CHECKS")
print("=" * 60)

print(
    "\nState Totals:",
    state_totals.duplicated(
        subset=["year", "state"]
    ).sum()
)

print(
    "State Industry:",
    state_industry.duplicated(
        subset=["year", "state", "industry_code"]
    ).sum()
)

print(
    "OEWS:",
    oews.duplicated(
        subset=["year", "state", "occupation_code"]
    ).sum()
)


# ==========================================
# BASIC VALUE CHECKS
# ==========================================

print("\n" + "=" * 60)
print("BASIC VALUE CHECKS")
print("=" * 60)

print(
    "\nNegative State Employment:",
    (state_totals["employment"] < 0).sum()
)

print(
    "Negative Industry Employment:",
    (state_industry["employment"] < 0).sum()
)

print(
    "Negative OEWS Employment:",
    (oews["employment"] < 0).sum()
)

print(
    "\nZero State Employment:",
    (state_totals["employment"] == 0).sum()
)

print(
    "Zero Industry Employment:",
    (state_industry["employment"] == 0).sum()
)

print(
    "Zero OEWS Employment:",
    (oews["employment"] == 0).sum()
)


STATE TOTALS
Shape: (561, 14)

Exact duplicate rows:
0

Data types:
year                                int64
area_fips                           int64
state                              object
employment                          int64
establishments                      int64
total_wages                         int64
avg_annual_pay                      int64
avg_weekly_wage                     int64
oty_annual_avg_emplvl_chg           int64
oty_annual_avg_emplvl_pct_chg     float64
oty_total_annual_wages_chg          int64
oty_total_annual_wages_pct_chg    float64
oty_avg_annual_pay_chg              int64
oty_avg_annual_pay_pct_chg        float64
dtype: object

Columns with missing values:
None

Year range:
2015 - 2025

STATE INDUSTRY
Shape: (11629, 11)

Exact duplicate rows:
0

Data types:
year                     int64
area_fips                int64
state                   object
industry_code           object
industry                object
employment               int64
establishm

In [18]:
# ==========================================
# FINAL CHECK
# Industry zero employment vs missing wages
# ==========================================

zero_emp = state_industry["employment"] == 0
missing_pay = state_industry["avg_annual_pay"].isna()
missing_weekly = state_industry["avg_weekly_wage"].isna()

print("Zero employment:", zero_emp.sum())
print("Missing annual pay:", missing_pay.sum())
print("Missing weekly wage:", missing_weekly.sum())

print(
    "\nZero employment AND missing annual pay:",
    (zero_emp & missing_pay).sum()
)

print(
    "Zero employment AND missing weekly wage:",
    (zero_emp & missing_weekly).sum()
)

print(
    "\nMissing annual pay with employment > 0:",
    (missing_pay & (state_industry["employment"] > 0)).sum()
)

print(
    "Missing weekly wage with employment > 0:",
    (missing_weekly & (state_industry["employment"] > 0)).sum()
)

Zero employment: 147
Missing annual pay: 147
Missing weekly wage: 147

Zero employment AND missing annual pay: 147
Zero employment AND missing weekly wage: 147

Missing annual pay with employment > 0: 0
Missing weekly wage with employment > 0: 0
